# Setup

In [2]:
!pip install packmol
!pip install py3Dmol

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.7 MB/s eta 0:00:00


In [4]:
!wget https://github.com/m3g/packmol/archive/refs/tags/v20.14.0.tar.gz
!tar -xzf v20.14.0.tar.gz
%cd packmol-20.14.0
!make

--2026-03-03 22:19:30--  https://github.com/m3g/packmol/archive/refs/tags/v20.14.0.tar.gz
Resolving github.com (github.com)... 140.82.116.4
Connecting to github.com (github.com)|140.82.116.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://codeload.github.com/m3g/packmol/tar.gz/refs/tags/v20.14.0 [following]
--2026-03-03 22:19:30--  https://codeload.github.com/m3g/packmol/tar.gz/refs/tags/v20.14.0
Resolving codeload.github.com (codeload.github.com)... 140.82.116.10
Connecting to codeload.github.com (codeload.github.com)|140.82.116.10|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/x-gzip]
Saving to: ‘v20.14.0.tar.gz’

v20.14.0.tar.gz         [ <=>                ]  98.59K  --.-KB/s    in 0.02s   

2026-03-03 22:19:31 (3.98 MB/s) - ‘v20.14.0.tar.gz’ saved [100953]

/content/packmol-20.14.0
 ------------------------------------------------------ 
 Compiling packmol with /usr/bin/gfortran 
 Flags: -O3

In [3]:
# Import Libraries
import packmol as mol
import numpy as np
import pandas as pd
import py3Dmol

In [ ]:
print_it = True

# Correct PC and DME Cubic Simulation

In [ ]:
# Attempting to get geometries already validated
%%bash
wget -q https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/propylene%20carbonate/SDF?record_type=3d -O pc.sdf
wget -q https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/dimethoxyethane/SDF?record_type=3d -O dme.sdf

In [ ]:
%%bash
apt-get -qq install openbabel
obabel pc.sdf -O pc.pdb
obabel dme.sdf -O dme.pdb

Selecting previously unselected package libboost-iostreams1.74.0:amd64.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../libboost-iostreams1.74.0_1.74.0-14ubuntu3_amd64.deb ...
Unpacking libboost-iostreams1.74.0:amd64 (1.74.0-14ubuntu3) ...
Selecting previously unselected package libinchi1.
Preparing to unpack .../libinchi1_1.03+dfsg-4_amd64.deb ...
Unpacking libinchi1 (1.03+dfsg-4) ...
Selecting previously unselected package libmaeparser1:amd64.
Preparing to unpack .../libmaeparser1_1.2.4-1build1_amd64.deb ...
Unpacking libmaeparser1:amd64 (1.2.4-1build1) ...
Selecting previously unselected package libopenbabel7.
Preparing to unpack .../libopenbabel7_3.1.1+dfsg-6ubuntu5_amd64.deb ...
Unpacking libopenbabel7 (3.1.1+dfsg-6ubuntu5) ...
Selecting previously unselected package openbabel.
Preparing to unpack .../openbabel_3.1.1+dfsg-6ubuntu5_amd64.deb ...
Unpacking openbabel (3.1.1+dfsg-6ubuntu5) ...
Setting up libboost-iostreams1.74.0:amd64 (

1 molecule converted
1 molecule converted


In [ ]:
def show(pdbfile):
    with open(pdbfile) as f:
        pdb = f.read()
    v = py3Dmol.view(width=300, height=300)
    v.addModel(pdb, "pdb")
    v.setStyle({"stick": {}})
    v.zoomTo()
    v.show()

show("pc.pdb")
show("dme.pdb")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
# Packmol input
%%bash
cat > input.inp << 'EOF'
tolerance 2.0
filetype pdb
output pc_dme_box_30A.pdb

structure pc.pdb
  number 102
  inside box 0. 0. 0. 30. 30. 30.
end structure

structure dme.pdb
  number 102
  inside box 0. 0. 0. 30. 30. 30.
end structure
EOF

In [ ]:
# Run packmol
!packmol < input.inp


################################################################################

 PACKMOL - Packing optimization for the automated generation of
 starting configurations for molecular dynamics simulations.
 
                                                             Version 21.2.1 

################################################################################

  Packmol must be run with: packmol < inputfile.inp 

  Userguide at: http://m3g.iqm.unicamp.br/packmol 

  Reading input file... (Control-C aborts)
  Types of coordinate files specified: pdb
  Seed for random number generator:      1234567
  Output file: pc_dme_box_30A.pdb
  Reading coordinate file: pc.pdb
  Reading coordinate file: dme.pdb
  Number of independent structures:            2
  The structures are: 
  Structure            1 :pc.pdb(          13  atoms)
  Structure            2 :dme.pdb(          16  atoms)
  Maximum number of GENCAN loops for all molecule packing:          400
  Distance tolerance:    2.000000

In [ ]:
with open("pc_dme_box_30A.pdb") as f:
    pdb = f.read()

view = py3Dmol.view(width=800, height=600)
view.addModel(pdb, "pdb")
view.setStyle({"stick": {}})
view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

# NaFP6

We are using $L = 30 \mathring{\mathrm A}$ so the volume of the box is $V=30^3=27000 \mathring {\mathrm A}^3$.

The Molar mass of $NaFP_6$ is $227.8307 \frac{g}{mol}$

The formula to find the number of molecules we need is
$$N=\frac{\rho V N_A}{M}$$

So we can plug in $\rho = 2.369\frac{g}{cm^3}$ (density), $V = 2.7\times 10^{-20}cm^3$ (box volume), $N_A = 6.022\times10^{23}$ (Avagadro's Number), and $M = 227.8307 \frac{g}{mol}$ (Molar Mass).

$$N=\frac{2.369 \cdot 2.7\times10^{-20}\cdot6.022\times10^{23}}{227.8307}$$

$N\approx 170$, where I rounded from 169.87 to 170 since the number is closer to 170 than 169.

For this one, I put the separated charged species of $Na^+$ and $PF_{6}^-$, so that is 170 of each.

In [16]:
#Making Na+ Manually
%%bash
cat > na.pdb << EOF
ATOM      1  NA  NA     1       0.000   0.000   0.000
END
EOF

In [17]:
# Get PF6- Geometry
%%bash
wget -q "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/CID/28013/SDF?record_type=3d" -O pf6.sdf
obabel pf6.sdf -O pf6.pdb
ls -lh pf6.pdb

-rw-r--r-- 1 root root 4.5K Mar  3 22:28 pf6.pdb


1 molecule converted


In [11]:
%%bash

apt-get -qq install openbabel

# Download validated 3D geometries from PubChem
wget -q "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/sodium%20ion/SDF?record_type=3d" -O na.sdf
wget -q "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/hexafluorophosphate/SDF?record_type=3d" -O pf6.sdf

# Convert to PDB
obabel na.sdf -O na.pdb
obabel pf6.sdf -O pf6.pdb

echo "Files created:"
ls -lh na.pdb pf6.pdb

Selecting previously unselected package libboost-iostreams1.74.0:amd64.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../libboost-iostreams1.74.0_1.74.0-14ubuntu3_amd64.deb ...
Unpacking libboost-iostreams1.74.0:amd64 (1.74.0-14ubuntu3) ...
Selecting previously unselected package libinchi1.
Preparing to unpack .../libinchi1_1.03+dfsg-4_amd64.deb ...
Unpacking libinchi1 (1.03+dfsg-4) ...
Selecting previously unselected package libmaeparser1:amd64.
Preparing to unpack .../libmaeparser1_1.2.4-1build1_amd64.deb ...
Unpacking libmaeparser1:amd64 (1.2.4-1build1) ...
Selecting previously unselected package libopenbabel7.
Preparing to unpack .../libopenbabel7_3.1.1+dfsg-6ubuntu5_amd64.deb ...
Unpacking libopenbabel7 (3.1.1+dfsg-6ubuntu5) ...
Selecting previously unselected package openbabel.
Preparing to unpack .../openbabel_3.1.1+dfsg-6ubuntu5_amd64.deb ...
Unpacking openbabel (3.1.1+dfsg-6ubuntu5) ...
Setting up libboost-iostreams1.74.0:amd64 (

0 molecules converted
0 molecules converted


In [18]:
# Verify The files exist, before using Packmol
%%bash
ls -lh na.pdb pf6.pdb

-rw-r--r-- 1 root root   58 Mar  3 22:27 na.pdb
-rw-r--r-- 1 root root 4.5K Mar  3 22:28 pf6.pdb


In [19]:
def show(pdbfile):
    with open(pdbfile) as f:
        pdb = f.read()
    v = py3Dmol.view(width=300, height=300)
    v.addModel(pdb, "pdb")
    v.setStyle({"stick": {}})
    v.zoomTo()
    v.show()

show("na.pdb") # Wont see Na+ because its a sodium ion with no internal geometry
show("pf6.pdb")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [20]:
# Create Packmol input (The 30 Angstrom Box)
%%bash
cat > input_nafp6.inp << 'EOF'
tolerance 2.0
filetype pdb
output nafp6_box_30A.pdb

structure na.pdb
  number 170
  inside box 0. 0. 0. 30. 30. 30.
end structure

structure pf6.pdb
  number 170
  inside box 0. 0. 0. 30. 30. 30.
end structure
EOF

In [21]:
# Run Packmol
!packmol < input_nafp6.inp

Streaming output truncated to the last 5000 lines.
  Maximum violation of the constraints: .70362E+01

--------------------------------------------------------------------------------


--------------------------------------------------------------------------------

  Starting GENCAN loop:          177
  Scaling radii by:    1.0000000000000000     

  Packing:|0                                                        100%|
          |*************************************************************|

  Function value from last GENCAN loop: f = .41863E+04
  Best function value before: f = .29469E+04
  Improvement from best function value:   -42.06 %
  Improvement from last loop:    16.49 %
  Maximum violation of target distance:     3.797017
  Maximum violation of the constraints: .67254E+01

--------------------------------------------------------------------------------


--------------------------------------------------------------------------------

  Starting GENCAN loop:          178

In [22]:
# Visualize Final Packed Box
with open("nafp6_box_30A.pdb") as f:
    pdb = f.read()

view = py3Dmol.view(width=800, height=600)
view.addModel(pdb, "pdb")
view.setStyle({"stick": {}})
view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.